In [2]:
import numpy as np

PATH_TRAIN = 'data/processed/new_handmarks.npy'
PATH_TEST = 'data/processed/new_targets.npy'

X = np.load(PATH_TRAIN, allow_pickle=True)
y = np.load(PATH_TEST, allow_pickle=True)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_2, y, test_size=0.15, 
                                                    stratify = y, random_state=42)


In [22]:
X_train

array([[[ 0.169     ,  0.84299999,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.177     ,  0.84200001,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        [ 0.17399999,  0.833     ,  0.        , ...,  0.        ,
          0.        ,  0.        ],
        ...,
        [ 0.68699998,  0.58899999, -0.        , ..., -0.186     ,
          0.        ,  0.        ],
        [ 0.68699998,  0.58999997, -0.        , ..., -0.189     ,
          0.        ,  0.        ],
        [ 0.68699998,  0.59100002, -0.        , ..., -0.18799999,
          0.        ,  0.        ]],

       [[ 0.89099997,  0.57300001,  0.        , ..., -0.043     ,
          0.        ,  0.        ],
        [ 0.89899999,  0.51200002,  0.        , ..., -0.009     ,
          0.        ,  0.        ],
        [ 0.84299999,  0.48800001,  0.        , ...,  0.011     ,
          0.        ,  0.        ],
        ...,
        [ 0.74400002,  0.479     , -0.        , ...,  

In [23]:
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler(feature_range=(-1, 1))

X_1d = X_train.reshape(-1)
X_2d = X_1d.reshape(-1, 1)  # Convert to 2D: (n_samples, 1)

mms.fit(X_2d)

MinMaxScaler(feature_range=(-1, 1))

In [28]:
#HERE I JUST PREPROCESS ALL FRAMES AND VIDEOS -
#THE IS A PADDING FOR VIDEOS AND FRAME; VIDEOS CONSIST OF 25 FRAMES SPACED ACCROSS VIDEO
#I ALSO DO SCALING (-1, 1) FOR TRAIN AND TEST DATA
import numpy as np


N_FRAMES = 25

def into_normal_arrays_with_padding(X_id_):
    X_train_processed = []
    for video in X_id_:
        k = max(1, len(video) // N_FRAMES)
        sampled_frames = []
        for i in range(0, len(video), k):
            if len(sampled_frames) >= N_FRAMES:
                break
            frame = mms.transform(np.array(video[i]).reshape(-1, 1))
            frame = frame.reshape(-1)
            if len(frame) < 128:
                frame = frame + [0.0] * (128 - len(frame))
            elif len(frame) > 128:
                frame = frame[:128]
            sampled_frames.append(np.array(frame, dtype=np.float32))
        while len(sampled_frames) < N_FRAMES:
            sampled_frames.append([0.0] * 128)
        X_train_processed.append(np.array(sampled_frames))
    return np.array(X_train_processed)
        

X_train_processed = into_normal_arrays_with_padding(X_train)
X_test_processed = into_normal_arrays_with_padding(X_train)
print(f"Final shape: {X_train_processed.shape}")

Final shape: (17000, 25, 128)


In [20]:
print(min(X_train[1][0]), max(X_train[1][0]))

[0.169, 0.843, 0.0, 0.243, 0.842, -0.016, 0.299, 0.851, -0.04, 0.333, 0.871, 
 -0.061, 0.352, 0.894, -0.083, 0.253, 0.837, -0.065, 0.265, 0.863, -0.101, 0.296, 
 0.886, -0.127, 0.32, 0.903, -0.144, 0.209, 0.85, -0.075, 0.227, 0.886, -0.11, 0.263,
 0.911, -0.122, 0.293, 0.926, -0.133, 0.172, 0.864, -0.085, 0.192, 0.902, -0.12, 0.227, 
 0.924, -0.123, 0.254, 0.937, -0.123, 0.139, 0.881, -0.096, 0.163, 0.915, -0.12, 0.196, 
 0.933, -0.121, 0.223, 0.942, -0.121]  #21 * 3 -> frame X_train[x][y]
                                        #frame * n (кадров) -> video X_train[x]
                                        #video * k -> X_train

-0.046 0.891


[0.169,
 0.843,
 0.0,
 0.243,
 0.842,
 -0.016,
 0.299,
 0.851,
 -0.04,
 0.333,
 0.871,
 -0.061,
 0.352,
 0.894,
 -0.083,
 0.253,
 0.837,
 -0.065,
 0.265,
 0.863,
 -0.101,
 0.296,
 0.886,
 -0.127,
 0.32,
 0.903,
 -0.144,
 0.209,
 0.85,
 -0.075,
 0.227,
 0.886,
 -0.11,
 0.263,
 0.911,
 -0.122,
 0.293,
 0.926,
 -0.133,
 0.172,
 0.864,
 -0.085,
 0.192,
 0.902,
 -0.12,
 0.227,
 0.924,
 -0.123,
 0.254,
 0.937,
 -0.123,
 0.139,
 0.881,
 -0.096,
 0.163,
 0.915,
 -0.12,
 0.196,
 0.933,
 -0.121,
 0.223,
 0.942,
 -0.121]

In [19]:
#WORK ON DECOMPOSITION LATER
from sklearn.random_projection import GaussianRandomProjection
from sklearn.pipeline import Pipeline

grp = GaussianRandomProjection(
    n_components = 0.9,
    copy = True,
    eps = 0.1,
    random_state = 42
)

[0.891,
 0.573,
 0.0,
 0.874,
 0.546,
 -0.003,
 0.836,
 0.53,
 -0.009,
 0.8,
 0.523,
 -0.012,
 0.779,
 0.513,
 -0.015,
 0.793,
 0.545,
 -0.032,
 0.737,
 0.544,
 -0.043,
 0.714,
 0.546,
 -0.044,
 0.707,
 0.548,
 -0.043,
 0.787,
 0.567,
 -0.033,
 0.718,
 0.572,
 -0.044,
 0.703,
 0.573,
 -0.042,
 0.706,
 0.575,
 -0.038,
 0.789,
 0.588,
 -0.032,
 0.731,
 0.594,
 -0.039,
 0.73,
 0.591,
 -0.032,
 0.745,
 0.588,
 -0.026,
 0.801,
 0.605,
 -0.03,
 0.763,
 0.608,
 -0.033,
 0.766,
 0.604,
 -0.025,
 0.779,
 0.6,
 -0.017,
 0.13,
 0.556,
 0.0,
 0.151,
 0.528,
 -0.004,
 0.19,
 0.509,
 -0.009,
 0.221,
 0.494,
 -0.014,
 0.235,
 0.481,
 -0.02,
 0.235,
 0.519,
 -0.016,
 0.294,
 0.515,
 -0.027,
 0.33,
 0.514,
 -0.033,
 0.357,
 0.514,
 -0.037,
 0.241,
 0.535,
 -0.021,
 0.302,
 0.532,
 -0.028,
 0.34,
 0.531,
 -0.033,
 0.368,
 0.532,
 -0.037,
 0.238,
 0.552,
 -0.026,
 0.295,
 0.552,
 -0.034,
 0.331,
 0.552,
 -0.041,
 0.355,
 0.553,
 -0.046,
 0.229,
 0.569,
 -0.031,
 0.266,
 0.575,
 -0.038,
 0.29,
 0.579,
 -0